In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

sns.set_style('whitegrid')
print('Librairies importées avec succès')

Librairies importées avec succès


In [3]:
df = pd.read_csv("datascience_salaries.csv")

# Supprimer la colonne d'index inutile si présente
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

Shape : (1171, 6)
Colonnes : ['job_title', 'job_type', 'experience_level', 'location', 'salary_currency', 'salary']


,job_title,job_type,experience_level,location,salary_currency,salary
0,Data scientist,Full Time,Senior,New York City,USD,149000
1,Data scientist,Full Time,Senior,Boston,USD,120000
2,Data scientist,Full Time,Senior,London,USD,68000
3,Data scientist,Full Time,Senior,Boston,USD,120000
4,Data scientist,Full Time,Senior,New York City,USD,149000


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
scaler = MinMaxScaler()
df['salary_normalized'] = scaler.fit_transform(df[['salary']])

print(f"Salaire min : ${df['salary'].min():,.0f}  ->  normalisé : {df['salary_normalized'].min():.4f}")
print(f"Salaire max : ${df['salary'].max():,.0f}  ->  normalisé : {df['salary_normalized'].max():.4f}")
df[['salary', 'salary_normalized']].head(10)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#f7f9fc')

ax1.hist(df['salary'], bins=40, color='#4e9af1', edgecolor='white', alpha=0.85)
ax1.set_title('Distribution des Salaires (originaux)', fontweight='bold')
ax1.set_xlabel('Salaire (USD)')
ax1.set_ylabel('Fréquence')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

ax2.hist(df['salary_normalized'], bins=40, color='#3dc78f', edgecolor='white', alpha=0.85)
ax2.set_title('Distribution des Salaires (Min-Max normalisés)', fontweight='bold')
ax2.set_xlabel('Valeur normalisée [0 - 1]')
ax2.set_ylabel('Fréquence')

plt.suptitle('Avant vs Après Normalisation Min-Max', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
le = LabelEncoder()
df_encoded = df.copy()

cat_cols = ['job_title', 'job_type', 'experience_level', 'location', 'salary_currency']
for col in cat_cols:
    df_encoded[col + '_enc'] = le.fit_transform(df_encoded[col])

features = ['salary_normalized', 'job_title_enc', 'job_type_enc',
            'experience_level_enc', 'location_enc', 'salary_currency_enc']
X = df_encoded[features]
print('Variables utilisées pour la PCA :')
print(features)

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(X)

df['PCA1'] = pca_result[:, 0]
df['PCA2'] = pca_result[:, 1]

explained = pca.explained_variance_ratio_
print(f'Variance expliquée — PC1 : {explained[0]:.2%}')
print(f'Variance expliquée — PC2 : {explained[1]:.2%}')
print(f'Total                    : {sum(explained):.2%}')

In [ ]:
palette = {'Entry':'#4e9af1','Mid':'#f4a62a','Senior':'#3dc78f','Executive':'#e05c6b'}
order   = ['Entry', 'Mid', 'Senior', 'Executive']

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#f7f9fc')

for lvl in order:
    mask = df['experience_level'] == lvl
    ax.scatter(df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'],
               label=lvl, alpha=0.55, s=30, color=palette[lvl])

ax.set_title(f'ACP 2D — Variance expliquée : {sum(explained):.1%}', fontweight='bold', fontsize=13)
ax.set_xlabel(f'PC1  ({explained[0]:.1%})')
ax.set_ylabel(f'PC2  ({explained[1]:.1%})')
ax.legend(title="Niveau d'expérience", framealpha=0.8)
plt.tight_layout()
plt.show()

In [ ]:
order = ['Entry', 'Mid', 'Senior', 'Executive']

agg = (
    df.groupby('experience_level')['salary']
    .agg(
        Effectif      ='count',
        Salaire_Moyen ='mean',
        Salaire_Median='median',
        Salaire_Min   ='min',
        Salaire_Max   ='max'
    )
    .reindex(order)
    .reset_index()
)

for col in ['Salaire_Moyen','Salaire_Median','Salaire_Min','Salaire_Max']:
    agg[col] = agg[col].round(0).astype(int)

agg.rename(columns={'experience_level': "Niveau d'Expérience"}, inplace=True)
agg

In [ ]:
import numpy as np

palette = {'Entry':'#4e9af1','Mid':'#f4a62a','Senior':'#3dc78f','Executive':'#e05c6b'}
order   = ['Entry', 'Mid', 'Senior', 'Executive']
x = np.arange(len(order))
width = 0.38

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#f7f9fc')

bars1 = ax.bar(x - width/2, agg['Salaire_Moyen'],  width, label='Moyen',
               color=[palette[l] for l in order], alpha=0.9, edgecolor='white')
bars2 = ax.bar(x + width/2, agg['Salaire_Median'], width, label='Médian',
               color=[palette[l] for l in order], alpha=0.5, edgecolor='white')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 600,
            f"${bar.get_height():,.0f}", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_title("Salaire Moyen vs Médian par Niveau d'Expérience", fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(order, fontsize=11)
ax.set_ylabel('Salaire (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#f7f9fc')

data_by_level = [df[df['experience_level'] == lvl]['salary'].values for lvl in order]
bp = ax.boxplot(data_by_level, patch_artist=True, tick_labels=order,
                medianprops=dict(color='black', linewidth=2.5))

for patch, lvl in zip(bp['boxes'], order):
    patch.set_facecolor(palette[lvl])
    patch.set_alpha(0.75)

ax.set_title("Distribution des Salaires par Niveau d'Expérience", fontweight='bold', fontsize=13)
ax.set_ylabel('Salaire (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))
plt.tight_layout()
plt.show()